### 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

### 2. Load Data 

In [2]:
df = pd.read_csv('../data/processed/processed_data.csv')

print("Shape:", df.shape)
print(df.head())

Shape: (100139, 91)
          UserID                 Email            RegistrationDate  \
0  DERMAI_000000    user1825@gmail.com  2026-04-18 14:54:55.322730   
1  DERMAI_000001    user4507@gmail.com  2024-08-21 14:54:55.322730   
2  DERMAI_000002    user3658@gmail.com  2024-12-11 14:54:55.322730   
3  DERMAI_000003  user1680@hotmail.com  2025-09-21 14:54:55.322730   
4  DERMAI_000004    user8936@gmail.com  2024-07-13 14:54:55.322730   

                   LastActive   Age  Gender     SkinType  SkinConcerns  \
0  2026-05-05 14:54:55.763338  39.0  Female       Normal  Pigmentation   
1  2026-05-13 14:54:55.763338  32.0    Male  Combination  Pigmentation   
2  2026-04-29 14:54:55.763338  41.0  Female         Oily  Pigmentation   
3  2026-04-19 14:54:55.763338  52.0    Male          Dry  Pigmentation   
4  2026-04-26 14:54:55.763338  31.0  Female         Oily       Dryness   

   FitzpatrickScale       Allergies  ... SkinType_encoded Gender_encoded  \
0                 4             NaN  .

### 3. User Feature Engineering

In [3]:
user_feature_cols = [
    'Age', 'MonthlyBudget_USD', 'ProductEffectiveness_Score',
    'CustomerSatisfaction_pct', 'Total_Active_Ingredients',
    'Routine_Consistency', 'Ingredient_Diversity'
]

# keep only existing columns
user_feature_cols = [c for c in user_feature_cols if c in df.columns]

# aggregate per user (VERY IMPORTANT FIX)
user_features = df.groupby('UserID')[user_feature_cols].mean().fillna(0)

print("User matrix shape:", user_features.shape)

User matrix shape: (100000, 4)


### 4. Scaling

In [4]:
scaler = StandardScaler()
user_features_scaled = scaler.fit_transform(user_features)

### 5. Train KNN Model

In [5]:
n_neighbors = min(20, len(user_features))

knn_model = NearestNeighbors(
    n_neighbors=n_neighbors,
    metric='cosine'
)

knn_model.fit(user_features_scaled)

print("KNN trained successfully")

KNN trained successfully


### 6. Similar Users Function

In [6]:
def find_similar_users(user_index, n=5):
    user_vector = user_features_scaled[user_index:user_index+1]

    distances, indices = knn_model.kneighbors(user_vector, n_neighbors=n+1)

    return indices[0][1:], distances[0][1:]

### 7. Content-Based Scoring

In [7]:
def calculate_product_score(product, user_profile):
    score = 0

    # skin type match
    if user_profile.get('skin_type') in product.get('skin_types', []):
        score += 0.3
    elif 'All' in product.get('skin_types', []):
        score += 0.15

    # concern match
    user_concerns = user_profile.get('skin_concerns', [])
    product_concerns = product.get('concerns', [])

    if user_concerns:
        matches = sum(1 for c in user_concerns if c in product_concerns)
        score += 0.25 * (matches / len(user_concerns))

    # budget logic
    budget = user_profile.get('monthly_budget', 50)

    if product['tier'] == 'affordable' and budget < 50:
        score += 0.2
    elif product['tier'] == 'mid_range' and 50 <= budget <= 150:
        score += 0.2
    elif product['tier'] == 'luxury' and budget > 150:
        score += 0.2

    # rating boost
    score += (product['rating'] - 4) * 0.15

    return min(score, 1.0)

### 8. Hybrid Recommender

In [9]:
products_db = {
    "cleanser": [
        {"name": "Gentle Cleanser", "brand": "CeraVe", "price": 10, "rating": 4.5},
        {"name": "Foaming Face Wash", "brand": "Neutrogena", "price": 8, "rating": 4.2}
    ]
}

def calculate_product_score(product, user_profile):
    return product["rating"]

class HybridRecommender:
    def __init__(self, products_db):
        self.products_db = products_db

    def get_recommendations(self, user_profile, top_n=5):
        results = []

        for category, products in self.products_db.items():
            for product in products:
                score = calculate_product_score(product, user_profile)

                results.append({
                    "category": category,
                    "name": product["name"],
                    "brand": product["brand"],
                    "price": product["price"],
                    "rating": product["rating"],
                    "score": score
                })

        results = sorted(results, key=lambda x: x["score"], reverse=True)
        return results[:top_n]


recommender = HybridRecommender(products_db)

print(recommender.get_recommendations({"skin_type": "dry"}))

[{'category': 'cleanser', 'name': 'Gentle Cleanser', 'brand': 'CeraVe', 'price': 10, 'rating': 4.5, 'score': 4.5}, {'category': 'cleanser', 'name': 'Foaming Face Wash', 'brand': 'Neutrogena', 'price': 8, 'rating': 4.2, 'score': 4.2}]


### 9. Testing

In [10]:
test_user = {
    'skin_type': 'Oily',
    'skin_concerns': ['Acne', 'Large Pores'],
    'monthly_budget': 50
}

recs = recommender.get_recommendations(test_user)

for i, r in enumerate(recs, 1):
    print(i, r["name"], r["score"])

1 Gentle Cleanser 4.5
2 Foaming Face Wash 4.2


### 10. Model Saving

In [11]:
os.makedirs('../data/models', exist_ok=True)

joblib.dump(knn_model, '../data/models/knn_model.pkl')
joblib.dump(scaler, '../data/models/scaler.pkl')
joblib.dump(user_features, '../data/models/user_features.pkl')

print("Models saved successfully")

Models saved successfully
